# 04 Oxygen evolution (IA-OER-001)

**Requirement:** `IA-OER-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

## Governing equations


\(\mathrm{4\,OH^- \to O_2 + 2\,H_2O + 4\,e^-}\). Layouts: planar, submerged, interdigitated
trunk-and-projection, corrugated, serpentine, discrete arrays, spiral bifilar, pleated
(EP Claims 1, 3, 12, 32, 38, 70, 77). Default porous metal mesh + OER catalyst (Claim 11).
OER sits closer to iron than ORR on charge and is electrically isolatable (US FIGS. 5A-5B).


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path
setup_path()
from plant_sim.params import default_params, CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G
from plant_sim.plotting import apply_style, timeseries, xy_plot, FIGURES
apply_style()
P = default_params()
print("6 M KOH default", P.c_KOH_mol_m3/1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G,2), round(CAPACITY_MAGNETITE_MAH_G,2))
print("figures ->", FIGURES)


## Isolated component simulation

In [ ]:
from plant_sim.components.oer import OxygenEvolution, OER_LAYOUTS, LAYOUT_AREA_FACTOR
from plant_sim.components.base import integrate_component
print(OER_LAYOUTS)
oer = OxygenEvolution(P, layout="interdigitated")
t, y = integrate_component(lambda t,y: oer.rhs(t,y,{"I_oer_A": 20*P.I_discharge_100h_A, "T_K": P.T_ep_sim_K, "a_oh": 6.0, "a_h2o": 0.72, "p_O2_Pa": P.P_atm_Pa, "oer_isolated": 0.0}, {}), oer.y0(), (0, 800), n_eval=120)
timeseries(t/60, {"theta_cat": y[0], "n_O2": y[1], "bubbles": y[2]}, xlabel="t (min)", ylabel="states", title="OER catalyst and gas", callout="IA-OER-001", stem="nb04_oer")
xy_plot(np.arange(len(LAYOUT_AREA_FACTOR)), {"af": np.array(list(LAYOUT_AREA_FACTOR.values()))}, xlabel="layout index", ylabel="area factor", title="OER layouts", callout=str(list(LAYOUT_AREA_FACTOR)), stem="nb04_layouts")


## Verification against patents / SSS002

In [ ]:
print('Interdigitated shortens ionic path vs planar (EP [0050], FIGS. 19-21).')